In [ ]:
# Import statements
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import gc

In [ ]:
# create a text file called samplepaths.txt that contains the full path of all the samples to be processed in the dataset
# bash command used is - find /path/to/directory/ -type f -name "*.h5ad" > samplepaths.txt

In [ ]:

# input file containing the full paths to all of the samples
input_file = "path/to/samplepaths.txt"

# iterating through the file and reading all lines
with open(input_file, 'r') as file:
    paths = [line.strip() for line in file]
    
# creating a dictionary, the sample name will be the key and corrseponding adata will be the value
adatas = {}

for path in paths:
    key = path.split('/')[-1].split('_')[0]
    adatas[key] = sc.read_h5ad(path)

# merging all the adatas
merged_adata = ad.concat(adatas, label="dataset_name", join="outer", fill_value=None)


# merged_adata is the merged anndata object, printing a summary of the structure
merged_adata


In [ ]:
merged_adata.obs

In [ ]:
# adding dataset_id column to obs
# merged_adata.obs['dataset_id'] = input_file.strip().split("/")[-3]
print(merged_adata)

In [ ]:
# # # Identify mitochondrial genes - this step is not needed for scpc samples as there is already a calculation done for MT genes
merged_adata.var['mt'] = merged_adata.var_names.str.startswith('MT-')

# # Calculate percentage of mitochondrial counts per cell
merged_adata.obs['percent_mito'] = (merged_adata[:, merged_adata.var['mt']].X.sum(axis=1) / merged_adata.X.sum(axis=1)) * 100
merged_adata.var

In [ ]:
# Look at the range of values in adata.X - to check whether the data is normalized or not
print(f"Min value: {merged_adata.X.min()}")
print(f"Max value: {merged_adata.X.max()}")


In [ ]:
merged_adata.obs

In [ ]:
# filtering and quality control
sc.pp.filter_cells(merged_adata, min_counts=1000) 
sc.pp.filter_cells(merged_adata, min_genes=200) 

merged_adata = merged_adata[merged_adata.obs['percent_mito'] < 20]

merged_adata

In [ ]:
# included columns
exclude_columns = [ "add the columns you want to inlcude the the adata.obs"]
for x in list(merged_adata.obs.columns):
    if x not in exclude_columns:
        del merged_adata.obs[x] 
   

# making observation and variable names unique
merged_adata.obs_names_make_unique()
merged_adata.var_names_make_unique()

In [ ]:
# Copy the raw un-normalized counts to a new layer - optional but recommended
# merged_adata.layers["counts"] = merged_adata.X.copy()

In [ ]:
merged_adata.obs

In [ ]:
merged_adata.var

In [ ]:
# convert to a sparse matrix for more efficent storage
from scipy.sparse import csr_matrix  
merged_adata.X = csr_matrix(merged_adata.X) 

In [ ]:
# generate a merged h5ad file
merged_adata.write("path to save merged h5ad file to")

In [ ]:
# manual gc
gc.collect()